# LACT / simtelarray 事件检查

这个 notebook 按物理流程检查一个事件：读取事件、准备 raw image、图像清理、Hillas 参数、方向/芯位重建、SDP 平面图。

不同输入格式只在第一个配置 cell 里切换；读出来以后，后面的分析和画图步骤保持一致。

In [ ]:
from pathlib import Path
import os

os.environ.setdefault("MPLCONFIGDIR", "/home/lhaaso/huangyiyun/tmp/matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

LACT_ROOT_FILE = Path("/home/lhaaso/huangyiyun/LACT/Sim_program/LACT_sim/run_logs/lact_root_only_full_response/lact_events.root")
if not LACT_ROOT_FILE.exists():
    LACT_ROOT_FILE = Path("/home/lhaaso/huangyiyun/LACT/Sim_program/LACT_sim/run_logs/lact_root_full_response/lact_events.root")

SIMTEL_FILE = Path("/eos/lhaaso/simulation/lactmc/prod1/point_gamma/simtel/zenith_20/azimuth_0/100GeV_1000GeV/run000001/lact_prod1_simtel_particle_gamma_energy_100.0_1000.0_zenith_20.0_azimuth_0.0_run_1_event_0.zst")
ROOT_EVENT_FILE = Path("/path/to/pylast_native.root")

# 只改这里即可："lact_root" / "simtel" / "root_event"
INPUT_KIND = "lact_root"
INPUT_FILE = LACT_ROOT_FILE
# INPUT_KIND = "simtel"; INPUT_FILE = SIMTEL_FILE
# INPUT_KIND = "root_event"; INPUT_FILE = ROOT_EVENT_FILE

EVENT_INDEX = 0
MAX_EVENTS = 10

print("input kind:", INPUT_KIND)
print("input file:", INPUT_FILE)
print("exists:", INPUT_FILE.exists())


In [ ]:
import time
import numpy as np
from IPython.display import display

from pylast.io import LactEventSource, SimtelEventSource, RootEventSource
from pylast.calib import Calibrator
from pylast.image import ImageProcessor
from pylast.reco import ShowerProcessor
from pylast.visualize import (
    EventVisualizer,
    hillas_parameter_rows,
    plot_clean_images,
    plot_event_cores,
    plot_event_sdp_planes,
    plot_event_sdp_planes_3d,
    plot_event_sdp_planes_3d_interactive,
    plot_raw_images,
    reconstruction_summary,
)


def step(name, fn):
    print(f"BEGIN {name}", flush=True)
    t0 = time.perf_counter()
    out = fn()
    print(f"END {name}: {time.perf_counter() - t0:.3f}s", flush=True)
    return out


def read_one_event(kind, filename, event_index=0, max_events=10):
    filename = str(filename)
    if kind == "lact_root":
        source = LactEventSource(filename, max_events=max_events)
        event = source[event_index]
    elif kind == "simtel":
        source = SimtelEventSource(filename, max_events=max(max_events, event_index + 1))
        events = [event for event in source]
        event = events[event_index]
    elif kind == "root_event":
        source = RootEventSource(filename, max_events=max_events)
        event = source[event_index]
    else:
        raise ValueError("INPUT_KIND must be lact_root, simtel, or root_event")
    return source, event, EventVisualizer(source)


def prepare_raw_image(source, event):
    # simtelarray 读出来通常还没有 raw image，需要先由 waveform 抽取；LACT ROOT 通常已经有。
    if getattr(event, "dl0", None) is None:
        calibrator = step("准备 raw image", lambda: Calibrator(source.subarray))
        step("抽取 raw image", lambda: calibrator(event))
    return event


def run_cleaning(source, event):
    processor = step("初始化图像清理", lambda: ImageProcessor(source.subarray))
    step("图像清理 + Hillas 参数", lambda: processor(event))
    return event


def run_reconstruction(source, event):
    config = """{
      "ShowerProcessor": {
        "GeometryReconstructionTypes": ["HillasReconstructor"],
        "HillasReconstructor": {
          "ImageQuery": "hillas_intensity > 100 && leakage_intensity_width_2 < 0.3"
        }
      }
    }
    """
    processor = step("初始化方向/芯位重建", lambda: ShowerProcessor(source.subarray, config_str=config))
    step("方向/芯位重建", lambda: processor(event))
    return event


def print_event_summary(event):
    shower = event.simulation.shower
    triggered = list(getattr(event.simulation, "triggered_tels", []))
    print("event_id:", event.event_id)
    print("run_id:", event.run_id)
    print("energy [TeV]:", float(shower.energy))
    print("true zenith [deg]:", 90.0 - np.degrees(float(shower.alt)))
    print("true azimuth [deg]:", np.degrees(float(shower.az)))
    print("true core [m]:", float(shower.core_x), float(shower.core_y))
    print("triggered telescopes:", triggered)


## 1. 读取事件

这里读入一种格式。后面的 cell 不再区分输入来自 LACT ROOT、simtelarray，还是 pylast 原生 ROOT。

In [ ]:
source, event, visualizer = step(
    "读取事件",
    lambda: read_one_event(INPUT_KIND, INPUT_FILE, EVENT_INDEX, MAX_EVENTS),
)
prepare_raw_image(source, event)
print_event_summary(event)


## 2. 望远镜分布、芯位和 SDP 投影

这里画阵列背景、真实芯位、触发望远镜，以及 SDP 平面在地面的投影。

In [ ]:
core_result = plot_event_cores(event, visualizer=visualizer, include_non_triggered=False)
display(core_result["figure"])

sdp_result = plot_event_sdp_planes(event, visualizer=visualizer, include_non_triggered=False)
display(sdp_result["figure"])


## 3. Raw image

这里画清理前的相机图像。对 LACT ROOT，这是输出里的积分 p.e.；对 simtelarray，这是从 waveform 抽取出的积分图像。

In [ ]:
raw_result = plot_raw_images(event, visualizer=visualizer, include_non_triggered=False)
display(raw_result["figure"])


## 4. Clean image 和 Hillas 参数

这里运行图像清理，并画清理后的图像和 Hillas 椭圆。

In [ ]:
print("clean 前触发望远镜:", list(getattr(event.simulation, "triggered_tels", [])))
run_cleaning(source, event)
print("clean 后触发望远镜:", list(getattr(event.simulation, "triggered_tels", [])))

rows = hillas_parameter_rows(event)
print("有 Hillas 参数的望远镜:", [row["tel_id"] for row in rows])
for row in rows:
    print(
        f"Tel {row['tel_id']:2d}: intensity={row['intensity']:.2f}, "
        f"length={row['length_rad']:.5g} rad, width={row['width_rad']:.5g} rad, "
        f"psi={np.degrees(row['psi_rad']):.2f} deg"
    )

clean_result = plot_clean_images(event, visualizer=visualizer, show_hillas=True, include_non_triggered=False)
display(clean_result["figure"])


## 5. 方向和芯位重建

这里用 pylast 的 Hillas 重建。若 `is_valid=False`，通常说明通过筛选的望远镜不够或图像质量条件太严。

In [ ]:
run_reconstruction(source, event)
summary = reconstruction_summary(event, "HillasReconstructor")
for key, value in summary.items():
    if isinstance(value, float):
        print(f"{key}: {value:.6g}")
    else:
        print(f"{key}: {value}")


## 6. 3D SDP 平面

这里在重建之后画 3D SDP。默认先画 Plotly 交互图，可以在 notebook 或导出的 HTML 里旋转、缩放；红色是真实芯位和真实方向，蓝色是重建芯位和重建方向。

In [ ]:
sdp_3d_interactive_figure = plot_event_sdp_planes_3d_interactive(
    event,
    visualizer=visualizer,
    include_non_triggered=False,
    z_max=1200.0,
    show_reco=True,
)
sdp_3d_interactive_figure

# 静态版本用于保存 PNG 或快速预览。
sdp_3d_result = plot_event_sdp_planes_3d(
    event,
    visualizer=visualizer,
    include_non_triggered=False,
    z_max=1200.0,
    show_reco=True,
    show=False,
)
display(sdp_3d_result["figure"])


## 7. 保存图片

可选。保存的文件名使用物理含义：core、sdp、raw、clean、sdp_3d。交互 3D 额外保存为 HTML。

In [ ]:
OUTPUT_DIR = Path(INPUT_FILE).parent / "pylast_visualize"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plots = {
    "core": core_result,
    "sdp": sdp_result,
    "raw": raw_result,
    "clean": clean_result,
    "sdp_3d": sdp_3d_result,
}
for name, result in plots.items():
    fig = result.get("figure")
    if fig is None:
        continue
    path = OUTPUT_DIR / f"event_{event.event_id}_{name}.png"
    fig.savefig(path, dpi=200, bbox_inches="tight")
    print(path)

html_path = OUTPUT_DIR / f"event_{event.event_id}_sdp_3d_interactive.html"
sdp_3d_interactive_figure.write_html(html_path, include_plotlyjs="cdn", full_html=True)
print(html_path)


## 服务器更新

```bash
cd /home/lhaaso/huangyiyun/LACT/Sim_program/pylast
git pull yun lact_sim
python -m pip install -e . --no-build-isolation
```

LACT_sim 仓库里的 notebook 副本：

```bash
cd /home/lhaaso/huangyiyun/LACT/Sim_program/LACT_sim
git pull
```
